<a href="https://colab.research.google.com/github/aljeralmazan28-web/Algoverse-Research-TNAM/blob/main/deepcdr._baseline3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sex-stratified multi-omics drug response (DeepCDR-style) — Colab notebook

**What this does:** builds a master (cell line × drug) table from GDSC + DepMap, infers sex from XIST/Y-gene expression, makes cell-/drug-/tissue-blind splits, trains PyTorch fusion models (early / late / attention / DeepCDR-like) with optional sex-aware components, and reports metrics separately for female and male lines.

**Put these files in `MyDrive/sex_drug_response/data/` before running:**

| File | Where from |
|---|---|
| `GDSC1_fitted_dose_response_27Oct23.xlsx`, `GDSC2_fitted_dose_response_27Oct23.xlsx` | you already have these |
| `1_Drug_listMon_Jun_24_09_00_55_2019.csv`, `drug_graph_feat.zip` | you already have these (DeepCDR) |
| `Model.csv` | DepMap portal → Downloads → "All Downloads" → pick a release |
| `OmicsExpressionProteinCodingGenesTPMLogp1.csv` | same release |
| `OmicsCNGene.csv` | same release |
| `OmicsSomaticMutationsMatrixHotspot.csv` | same release |
| *(optional)* proteomics matrix CSV | e.g. the Gygi/Nusinow CCLE proteomics; rows = cell lines, columns = proteins |

File names in DepMap releases change between versions. If yours differ, edit the `FILES` dict in cell 2.

## 1. Setup

In [ ]:
!pip -q install hickle pyarrow openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, zipfile, random, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split, KFold
from sklearn.mixture import GaussianMixture
warnings.filterwarnings('ignore')

BASE  = Path('/content/drive/MyDrive/sex_drug_response')
DATA  = BASE / 'data'
CACHE = BASE / 'cache'
OUT   = BASE / 'outputs'
for p in (DATA, CACHE, OUT): p.mkdir(parents=True, exist_ok=True)

FILES = dict(
    gdsc1    = 'GDSC1_fitted_dose_response_27Oct23.xlsx',
    gdsc2    = 'GDSC2_fitted_dose_response_27Oct23.xlsx',
    druglist = '1_Drug_listMon_Jun_24_09_00_55_2019.csv',
    graphzip = 'drug_graph_feat.zip',
    model    = 'Model.csv',
    expr     = 'OmicsExpressionProteinCodingGenesTPMLogp1.csv',
    cn       = 'OmicsCNGene.csv',
    mut      = 'OmicsSomaticMutationsMatrixHotspot.csv',
    prot     = None,   # e.g. 'proteomics.csv' (optional)
)
GDSC_SET   = 'gdsc2'   # 'gdsc1' or 'gdsc2'. Different assays: do NOT pool blindly.
TARGET     = 'AUC'     # or 'LN_IC50'
MAX_ATOMS  = 100       # DeepCDR's padding size for drug graphs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

missing = [k for k, v in FILES.items() if v and not (DATA / v).exists()]
print('missing files:', missing if missing else 'none')

## 2. GDSC drug-response labels
The 2023 GDSC files are keyed by `SANGER_MODEL_ID` (SIDM...), not COSMIC ID. We bridge to DepMap through `Model.csv`.

In [ ]:
def load_gdsc(key):
    pq = CACHE / f'{key}.parquet'
    if pq.exists():
        return pd.read_parquet(pq)
    df = pd.read_excel(DATA / FILES[key])          # ~1-2 min per file, cached afterwards
    df.to_parquet(pq)
    return df

gdsc = load_gdsc(GDSC_SET)
print(gdsc.shape)
print(gdsc.columns.tolist())
print('cell lines:', gdsc.SANGER_MODEL_ID.nunique(), '| drugs:', gdsc.DRUG_ID.nunique(),
      '| cancer types:', gdsc.CANCER_TYPE.nunique())
print('duplicate (cell, drug) pairs:', gdsc.duplicated(['SANGER_MODEL_ID', 'DRUG_ID']).sum())
gdsc[[TARGET]].describe()

## 3. Drug graphs (DeepCDR `.hkl` files) and GDSC ↔ PubChem mapping

In [ ]:
import hickle as hkl

gdir = CACHE / 'drug_graph_feat'
if not gdir.exists():
    with zipfile.ZipFile(DATA / FILES['graphzip']) as z:
        z.extractall(CACHE)
hkl_files = {int(p.stem): p for p in gdir.glob('*.hkl')}
print('graph files:', len(hkl_files))

# --- inspect ONE file so you can verify the format before trusting the loader ---
pid0 = sorted(hkl_files)[0]
obj = hkl.load(str(hkl_files[pid0]))
print('type:', type(obj), '| len:', len(obj))
feat_mat, adj_list, deg_list = obj[0], obj[1], obj[2]
print('feat_mat:', np.asarray(feat_mat).shape, '(expect n_atoms x 75)')
print('adj_list: len', len(adj_list), '| first entry:', adj_list[0])

In [ ]:
def norm_adj(a):
    a = a + np.eye(a.shape[0], dtype='float32')
    d = a.sum(1)
    dinv = 1.0 / np.sqrt(d)
    return a * dinv[:, None] * dinv[None, :]

def build_graph(feat_mat, adj_list):
    feat_mat = np.asarray(feat_mat, dtype='float32')
    n = len(adj_list)
    if n > MAX_ATOMS or feat_mat.shape[0] != n:
        return None
    feat = np.zeros((MAX_ATOMS, feat_mat.shape[1]), 'float32')
    feat[:n] = feat_mat
    A = np.zeros((n, n), 'float32')
    for i, nb in enumerate(adj_list):
        for j in nb:
            A[i, int(j)] = 1
    A = np.maximum(A, A.T)
    adj = np.zeros((MAX_ATOMS, MAX_ATOMS), 'float32')
    adj[:n, :n] = norm_adj(A)
    return feat, adj

dl = pd.read_csv(DATA / FILES['druglist'], encoding='latin1')
dl['pubchem'] = pd.to_numeric(dl['PubCHEM'], errors='coerce')       # 'several'/'none' -> NaN
print('drug-list rows without a usable PubChem ID:', dl.pubchem.isna().sum())

graphs = {}
for pid, path in hkl_files.items():
    o = hkl.load(str(path))
    g = build_graph(o[0], o[1])
    if g is not None:
        graphs[pid] = g
print('graphs built:', len(graphs), '/', len(hkl_files))

# GDSC DRUG_ID -> PubChem -> graph
dl_map = dl.dropna(subset=['pubchem']).assign(pubchem=lambda d: d.pubchem.astype(int))
dl_map = dl_map[dl_map.pubchem.isin(graphs)]
drug_ids_in_gdsc = set(gdsc.DRUG_ID.unique())
dl_map = dl_map[dl_map.drug_id.isin(drug_ids_in_gdsc)].drop_duplicates('drug_id').reset_index(drop=True)
print(f'GDSC drugs: {len(drug_ids_in_gdsc)} -> with a DeepCDR graph: {len(dl_map)}')

drug_index = {int(d): i for i, d in enumerate(dl_map.drug_id)}
DRUG_FEAT = np.stack([graphs[p][0] for p in dl_map.pubchem])    # (n_drugs, 100, 75)
DRUG_ADJ  = np.stack([graphs[p][1] for p in dl_map.pubchem])    # (n_drugs, 100, 100)
drug_table = dl_map[['drug_id', 'Name', 'Targets', 'Target pathway', 'pubchem']]
print(DRUG_FEAT.shape, DRUG_ADJ.shape)

## 4. DepMap: metadata, omics, and sex

In [ ]:
def load_depmap_matrix(path):
    """Read a DepMap omics CSV (rows = models). Handles both the old format (first column = ACH-id)
    and the newer one (extra columns such as SequencingID / IsDefaultEntryForModel)."""
    df = pd.read_csv(path, low_memory=False)
    if 'ModelID' in df.columns:
        if 'IsDefaultEntryForModel' in df.columns:
            df = df[df['IsDefaultEntryForModel'].astype(str).str.lower().isin(['yes', 'true'])]
        df = df.set_index('ModelID')
    else:
        df = df.set_index(df.columns[0])
    meta = ['SequencingID', 'ModelConditionID', 'IsDefaultEntryForModel', 'IsDefaultEntryForMC', 'Unnamed: 0']
    df = df.drop(columns=[c for c in meta if c in df.columns])
    df = df[~df.index.duplicated()]
    return df.apply(pd.to_numeric, errors='coerce').astype('float32')

def gene_name(c):               # 'TP53 (7157)' -> 'TP53'
    return c.split(' (')[0]

model = pd.read_csv(DATA / FILES['model'], low_memory=False)
print('Model.csv columns:', model.columns.tolist())

In [ ]:
# Pick columns defensively (names vary slightly between DepMap releases)
def pick(df, options):
    for o in options:
        if o in df.columns: return o
    raise KeyError(f'none of {options} in {df.columns.tolist()}')

C_MODEL  = pick(model, ['ModelID'])
C_SANGER = pick(model, ['SangerModelID'])
C_SEX    = pick(model, ['Sex', 'PatientSex'])
C_TISSUE = pick(model, ['OncotreeLineage', 'PrimaryDisease'])
C_CCLE   = pick(model, ['CCLEName', 'StrippedCellLineName'])

model = model.dropna(subset=[C_SANGER]).drop_duplicates(C_SANGER)
print('DepMap models with a Sanger ID:', len(model))
joined = gdsc.SANGER_MODEL_ID.isin(model[C_SANGER])
print(f'GDSC cell lines found in DepMap: {gdsc.loc[joined, "SANGER_MODEL_ID"].nunique()} / {gdsc.SANGER_MODEL_ID.nunique()}')

expr = load_depmap_matrix(DATA / FILES['expr']);  expr.columns = [gene_name(c) for c in expr.columns]
expr = expr.loc[:, ~expr.columns.duplicated()]
cn   = load_depmap_matrix(DATA / FILES['cn']);    cn.columns   = [gene_name(c) for c in cn.columns]
cn   = cn.loc[:, ~cn.columns.duplicated()]
mut  = load_depmap_matrix(DATA / FILES['mut']);   mut.columns  = [gene_name(c) for c in mut.columns]
mut  = (mut.fillna(0) > 0).astype('float32');     mut = mut.loc[:, ~mut.columns.duplicated()]
print('expr', expr.shape, '| cn', cn.shape, '| mut', mut.shape)

In [ ]:
# ---------- Sex inference from XIST + Y-linked genes ----------
Y_GENES = ['RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY', 'EIF1AY', 'ZFY', 'USP9Y', 'TXLNGY', 'NLGN4Y', 'TMSB4Y', 'PCDH11Y', 'SRY']
y_present = [g for g in Y_GENES if g in expr.columns]
print('Y genes found:', y_present, '| XIST present:', 'XIST' in expr.columns)
XY_GENES = (['XIST'] if 'XIST' in expr.columns else []) + y_present   # used for the "remove X/Y genes" ablation

sx = pd.DataFrame({'xist': expr['XIST'], 'ymean': expr[y_present].mean(axis=1)}).dropna()
Z = (sx - sx.mean()) / sx.std()
gmm = GaussianMixture(2, covariance_type='full', random_state=0).fit(Z.values)
male_comp = int(np.argmax(gmm.means_[:, 1]))                       # component with higher Y expression
sx['p_male'] = gmm.predict_proba(Z.values)[:, male_comp]
sx['sex_inferred'] = np.where(sx.p_male > 0.95, 'Male', np.where(sx.p_male < 0.05, 'Female', 'Ambiguous'))

cells_all = model.set_index(C_MODEL)[[C_SANGER, C_CCLE, C_SEX, C_TISSUE]].copy()
cells_all.columns = ['sanger_id', 'ccle_name', 'sex_meta', 'tissue']
cells_all['sex_meta'] = cells_all.sex_meta.replace({'Unknown': np.nan}).str.capitalize()
cells_all = cells_all.join(sx[['xist', 'ymean', 'p_male', 'sex_inferred']])

print(pd.crosstab(cells_all.sex_meta.fillna('missing'), cells_all.sex_inferred.fillna('no expr')))

# QC plot
fig, ax = plt.subplots(figsize=(6, 5))
for lab, col in [('Female', 'tab:red'), ('Male', 'tab:blue'), ('missing', 'gray')]:
    s = cells_all[cells_all.sex_meta.fillna('missing') == lab]
    ax.scatter(s.ymean, s.xist, s=8, alpha=.6, c=col, label=f'metadata: {lab}')
ax.set_xlabel('mean Y-gene expression (log TPM)'); ax.set_ylabel('XIST (log TPM)'); ax.legend()
plt.tight_layout(); plt.savefig(OUT / 'sex_inference_qc.png', dpi=150); plt.show()

In [ ]:
# Final sex label: metadata when present, else inferred (if confident). Keep both + mismatch flag.
def final_sex(r):
    if r.sex_meta in ('Female', 'Male'): return r.sex_meta
    if r.sex_inferred in ('Female', 'Male'): return r.sex_inferred
    return 'Unknown'

cells_all['sex'] = cells_all.apply(final_sex, axis=1)
cells_all['sex_mismatch'] = (cells_all.sex_meta.isin(['Female', 'Male']) &
                             cells_all.sex_inferred.isin(['Female', 'Male']) &
                             (cells_all.sex_meta != cells_all.sex_inferred))

# Hypothesis 1: mislabel rate and sex balance per tissue
known = cells_all[cells_all.sex_meta.isin(['Female', 'Male']) & cells_all.sex_inferred.isin(['Female', 'Male'])]
tissue_qc = (cells_all.groupby('tissue')
    .agg(n=('sex', 'size'),
         n_female=('sex', lambda s: (s == 'Female').sum()),
         n_male=('sex', lambda s: (s == 'Male').sum()),
         n_unknown=('sex', lambda s: (s == 'Unknown').sum()),
         meta_missing=('sex_meta', lambda s: s.isna().sum()))
    .join(known.groupby('tissue').sex_mismatch.agg(['sum', 'count']).rename(columns={'sum': 'mismatch', 'count': 'n_checked'}))
    .sort_values('n', ascending=False))
tissue_qc['mismatch_rate'] = tissue_qc.mismatch / tissue_qc.n_checked
tissue_qc.to_csv(OUT / 'tissue_sex_qc.csv')
print('overall mismatch rate:', round(known.sex_mismatch.mean(), 4), f'({known.sex_mismatch.sum()}/{len(known)})')
tissue_qc.head(25)

## 5. Master table: cells × layers × drugs
Top-variance feature selection (unsupervised) keeps the models small. Z-scoring is done later, on **training cell lines only**, to avoid leakage.

In [ ]:
N_EXPR, N_CN, N_MUT = 3000, 2000, 2000

def top_var(df, k, force=()):
    v = df.var(axis=0, skipna=True).sort_values(ascending=False)
    keep = list(v.index[:k])
    keep += [g for g in force if g in df.columns and g not in keep]     # always keep XIST / Y genes
    return df[keep]

expr_s = top_var(expr, N_EXPR, force=XY_GENES)
cn_s   = top_var(cn,   N_CN,   force=[g for g in XY_GENES if g in cn.columns])
mut_f  = mut.loc[:, mut.sum(0) >= 5]
mut_s  = mut_f[mut_f.sum(0).sort_values(ascending=False).index[:N_MUT]]

# Optional proteomics (rows = cell lines; index may be ACH- IDs or CCLE names)
prot_s = None
if FILES['prot']:
    p = pd.read_csv(DATA / FILES['prot'], index_col=0, low_memory=False).apply(pd.to_numeric, errors='coerce')
    if not str(p.index[0]).startswith('ACH-'):
        m = cells_all.reset_index().set_index('ccle_name')[C_MODEL]
        p.index = p.index.map(m.to_dict())
    p = p[p.index.notna() & ~p.index.duplicated()]
    p = p.loc[:, p.notna().mean() >= 0.7]
    prot_s = top_var(p, 2000)
    print('proteomics:', prot_s.shape)

# cells that appear in GDSC, have a Sanger ID in DepMap, and have expr+cn+mut
sanger2ach = cells_all.reset_index().set_index('sanger_id')[C_MODEL].to_dict()
gdsc_cells = {sanger2ach[s] for s in gdsc.SANGER_MODEL_ID.unique() if s in sanger2ach}
ach = [a for a in cells_all.index
       if a in gdsc_cells and a in expr_s.index and a in cn_s.index and a in mut_s.index]
print('cell lines in GDSC ∩ DepMap ∩ {expr, cn, mut}:', len(ach), '(of', gdsc.SANGER_MODEL_ID.nunique(), 'in GDSC)')

cells = cells_all.loc[ach].reset_index().rename(columns={C_MODEL: 'model_id'})
cells['cell_idx'] = np.arange(len(cells))
cells['tissue'] = cells.tissue.fillna('Unknown')
cells['has_prot'] = cells.model_id.isin(prot_s.index) if prot_s is not None else False

LAYERS = {'expr': expr_s.loc[ach], 'cn': cn_s.loc[ach], 'mut': mut_s.loc[ach]}
if prot_s is not None:
    LAYERS['prot'] = prot_s.reindex(ach)                               # NaN rows where missing
LAYER_NAMES = {k: list(v.columns) for k, v in LAYERS.items()}
LAYER_ARR   = {k: v.values.astype('float32') for k, v in LAYERS.items()}
print({k: v.shape for k, v in LAYER_ARR.items()})

# pairs
g = gdsc[gdsc.SANGER_MODEL_ID.isin(cells.sanger_id) & gdsc.DRUG_ID.isin(drug_index)].copy()
g = g.dropna(subset=[TARGET]).groupby(['SANGER_MODEL_ID', 'DRUG_ID'], as_index=False)[TARGET].mean()
cell_lookup = cells.set_index('sanger_id').cell_idx.to_dict()
pairs = pd.DataFrame({
    'cell_idx': g.SANGER_MODEL_ID.map(cell_lookup).astype(int),
    'drug_idx': g.DRUG_ID.map(drug_index).astype(int),
    'y':        g[TARGET].astype('float32')})
pairs['sex']    = cells.sex.values[pairs.cell_idx]
pairs['tissue'] = cells.tissue.values[pairs.cell_idx]
print('pairs:', len(pairs), '| cells:', pairs.cell_idx.nunique(), '| drugs:', pairs.drug_idx.nunique())
print(pairs.groupby('sex').agg(pairs=('y', 'size'), cells=('cell_idx', 'nunique')))

# join-loss report
print(f'\nJoin loss: GDSC cell lines {gdsc.SANGER_MODEL_ID.nunique()} -> {cells.shape[0]} usable; '
      f'GDSC drugs {gdsc.DRUG_ID.nunique()} -> {pairs.drug_idx.nunique()} with graphs')

cells.to_csv(OUT / 'cells_master.csv', index=False); pairs.to_parquet(OUT / 'pairs_master.parquet')
drug_table.to_csv(OUT / 'drugs_master.csv', index=False)

## 6. Splits (random / cell-blind / drug-blind / tissue-blind), saved to Drive
Cell-blind is stratified by sex × tissue. All models read the same split files.

In [ ]:
def strat_labels(c, min_n=4):
    lab = (c.sex + '|' + c.tissue).astype(str)
    cnt = lab.map(lab.value_counts())
    lab = lab.where(cnt >= min_n, 'rare')
    if (lab == 'rare').sum() < 2:                                    # a single 'rare' sample can't be stratified
        lab = lab.where(lab != 'rare', lab.value_counts().index[0])
    return lab

def tvt_cells(cell_ids, strata, rng, test_frac=0.2, val_frac=0.1):
    try:
        trv, te = train_test_split(cell_ids, test_size=test_frac, stratify=strata.loc[cell_ids], random_state=rng)
        tr, va  = train_test_split(trv, test_size=val_frac / (1 - test_frac), stratify=strata.loc[trv], random_state=rng)
    except ValueError:                                               # fall back if a stratum is too small
        trv, te = train_test_split(cell_ids, test_size=test_frac, random_state=rng)
        tr, va  = train_test_split(trv, test_size=val_frac / (1 - test_frac), random_state=rng)
    return tr, va, te

def make_splits(pairs, cells, seed=0):
    S = {}
    ci, di = pairs.cell_idx.values, pairs.drug_idx.values
    idx = np.arange(len(pairs))
    # random pairs (this is what the original DeepCDR did; easiest, least realistic)
    trv, te = train_test_split(idx, test_size=0.2, random_state=seed)
    tr, va  = train_test_split(trv, test_size=0.125, random_state=seed)
    S['random'] = (tr, va, te)
    # cell-blind, stratified by sex x tissue
    strata = pd.Series(strat_labels(cells).values, index=cells.cell_idx.values)
    uc = pairs.cell_idx.unique()
    a, b, c = tvt_cells(uc, strata, seed)
    S['cell_blind'] = tuple(idx[np.isin(ci, x)] for x in (a, b, c))
    # drug-blind
    ud = pairs.drug_idx.unique()
    trv_d, te_d = train_test_split(ud, test_size=0.2, random_state=seed)
    tr_d, va_d  = train_test_split(trv_d, test_size=0.125, random_state=seed)
    S['drug_blind'] = tuple(idx[np.isin(di, x)] for x in (tr_d, va_d, te_d))
    # tissue-blind: 5 folds over tissues (fold k = held-out tissues)
    tissues = np.array(sorted(pairs.tissue.unique()))
    for k, (_, te_t) in enumerate(KFold(5, shuffle=True, random_state=seed).split(tissues)):
        te_tissues = set(tissues[te_t])
        te_cells = cells.cell_idx[cells.tissue.isin(te_tissues)].values
        rest = np.setdiff1d(pairs.cell_idx.unique(), te_cells)
        tr_c, va_c = train_test_split(rest, test_size=0.1, random_state=seed)
        S[f'tissue_blind_{k}'] = (idx[np.isin(ci, tr_c)], idx[np.isin(ci, va_c)], idx[np.isin(ci, te_cells)])
    return S

SPLITS = make_splits(pairs, cells, seed=0)
np.savez(OUT / 'splits_seed0.npz', **{f'{k}_{n}': v for k, s in SPLITS.items() for n, v in zip(('train', 'val', 'test'), s)})

rows = []
for k, (tr, va, te) in SPLITS.items():
    t = pairs.iloc[te]
    rows.append(dict(split=k, train=len(tr), val=len(va), test=len(te),
                     test_cells=t.cell_idx.nunique(), test_female_cells=t[t.sex == 'Female'].cell_idx.nunique(),
                     test_male_cells=t[t.sex == 'Male'].cell_idx.nunique()))
pd.DataFrame(rows)

## 7. Models
Drug branch: 3-layer GCN on the DeepCDR graph (dense normalised adjacency). Omics branches + fusion are selectable. This is a PyTorch re-implementation, not the original Keras code.

In [ ]:
class DrugGCN(nn.Module):
    def __init__(self, in_dim=75, hid=128, out=100, n_layers=3):
        super().__init__()
        self.lins = nn.ModuleList([nn.Linear(in_dim if i == 0 else hid, hid) for i in range(n_layers)])
        self.out = nn.Linear(hid, out)
    def forward(self, x, adj):                    # x (B,100,75), adj (B,100,100); padded rows of adj are zero
        for lin in self.lins:
            x = F.relu(torch.bmm(adj, lin(x)))
        return F.relu(self.out(x.max(dim=1).values))

def mlp_enc(d_in, d_out=100, hid=512, p=0.2):
    return nn.Sequential(nn.Linear(d_in, hid), nn.LayerNorm(hid), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(hid, d_out), nn.ReLU())

class DRModel(nn.Module):
    """fusion in {'early','late','attention','deepcdr_like'}"""
    def __init__(self, layer_dims, fusion='late', sex_input=False, emb=100, tok=64):
        super().__init__()
        self.names, self.fusion, self.sex_input = list(layer_dims), fusion, sex_input
        self.drug = DrugGCN(out=emb)
        sex_d = 2 if sex_input else 0
        if fusion == 'early':
            self.enc = mlp_enc(sum(layer_dims.values()), emb)
            head_in = emb * 2 + sex_d
        elif fusion in ('late', 'deepcdr_like'):
            self.enc = nn.ModuleDict({k: mlp_enc(d, emb) for k, d in layer_dims.items()})
            head_in = emb * (len(layer_dims) + 1) + sex_d
            if fusion == 'deepcdr_like':
                self.conv = nn.Sequential(nn.Conv1d(1, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4),
                                          nn.Conv1d(16, 32, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4), nn.Flatten())
                with torch.no_grad():
                    head_in = self.conv(torch.zeros(1, 1, head_in)).shape[1]
        elif fusion == 'attention':
            self.enc = nn.ModuleDict({k: mlp_enc(d, emb) for k, d in layer_dims.items()})
            self.proj = nn.ModuleDict({k: nn.Linear(emb, tok) for k in list(layer_dims) + ['drug']})
            self.sex_proj = nn.Linear(2, tok) if sex_input else None
            n_tok = len(layer_dims) + 1 + (1 if sex_input else 0)
            self.type_emb = nn.Parameter(torch.randn(n_tok, tok) * 0.02)
            self.tf = nn.TransformerEncoder(
                nn.TransformerEncoderLayer(tok, 4, 128, dropout=0.1, batch_first=True), num_layers=2)
            head_in = tok
        else:
            raise ValueError(fusion)
        self.head = nn.Sequential(nn.Linear(head_in, 256), nn.ReLU(), nn.Dropout(0.2),
                                  nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, omics, sex, dfeat, dadj):
        d = self.drug(dfeat, dadj)
        if self.fusion == 'early':
            z = torch.cat([self.enc(torch.cat([omics[k] for k in self.names], 1)), d], 1)
            if self.sex_input: z = torch.cat([z, sex], 1)
        elif self.fusion in ('late', 'deepcdr_like'):
            z = torch.cat([self.enc[k](omics[k]) for k in self.names] + [d], 1)
            if self.sex_input: z = torch.cat([z, sex], 1)
            if self.fusion == 'deepcdr_like': z = self.conv(z.unsqueeze(1))
        else:
            toks = [self.proj[k](self.enc[k](omics[k])) for k in self.names] + [self.proj['drug'](d)]
            if self.sex_input: toks.append(self.sex_proj(sex))
            t = torch.stack(toks, 1) + self.type_emb
            z = self.tf(t).mean(1)
        return self.head(z).squeeze(-1)

## 8. Training + evaluation
Sex-aware options: `sex_input` (append sex), `reweight` (sex-balanced loss weights). Ablations: `shuffle_sex`, `drop_xy` (removes XIST + Y genes; X-linked genes need a chromosome annotation, see note at the end), `layers` (drop an omics layer).

In [ ]:
def metrics(y, p):
    if len(y) < 3: return dict(n=len(y), rmse=np.nan, pearson=np.nan, spearman=np.nan)
    return dict(n=len(y), rmse=float(np.sqrt(np.mean((y - p) ** 2))),
                pearson=float(pearsonr(y, p)[0]), spearman=float(spearmanr(y, p)[0]))

def sex_report(df):
    out = {'all': metrics(df.y.values, df.pred.values)}
    for s in ('Female', 'Male'):
        d = df[df.sex == s]; out[s] = metrics(d.y.values, d.pred.values)
    return out

def bootstrap_gap(df, n_boot=200, seed=0):
    """Bootstrap over CELL LINES (not pairs) for RMSE(F) - RMSE(M)."""
    rng = np.random.default_rng(seed)
    groups = {c: g for c, g in df.groupby('cell_idx')}
    by_sex = {s: [c for c, g in groups.items() if g.sex.iloc[0] == s] for s in ('Female', 'Male')}
    gaps = []
    for _ in range(n_boot):
        r = []
        for s in ('Female', 'Male'):
            ch = rng.choice(by_sex[s], len(by_sex[s]))
            d = pd.concat([groups[c] for c in ch])
            r.append(np.sqrt(np.mean((d.y - d.pred) ** 2)))
        gaps.append(r[0] - r[1])
    return float(np.mean(gaps)), float(np.percentile(gaps, 2.5)), float(np.percentile(gaps, 97.5))

# tensors that never change
T_DFEAT = torch.tensor(DRUG_FEAT, device=DEVICE)
T_DADJ  = torch.tensor(DRUG_ADJ,  device=DEVICE)
SEX_VEC = np.zeros((len(cells), 2), 'float32')
SEX_VEC[cells.sex.values == 'Female', 0] = 1
SEX_VEC[cells.sex.values == 'Male', 1]   = 1

DEFAULT_CFG = dict(name='run', fusion='late', layers=['expr', 'mut', 'cn'], sex_input=False, reweight=False,
                   shuffle_sex=False, drop_xy=False, epochs=40, bs=512, lr=1e-3, wd=1e-5, patience=6, train_frac=1.0)

def run_experiment(cfg, split_name, seed=0, verbose=True):
    cfg = {**DEFAULT_CFG, **cfg}
    set_seed(seed)
    tr, va, te = [np.asarray(x) for x in SPLITS[split_name]]

    # keep only cells that have every requested layer (matters for proteomics)
    ok = np.ones(len(cells), bool)
    if 'prot' in cfg['layers'] or cfg.get('require_prot'): ok &= cells.has_prot.values
    ok_pair = ok[pairs.cell_idx.values]
    tr, va, te = tr[ok_pair[tr]], va[ok_pair[va]], te[ok_pair[te]]
    if cfg['train_frac'] < 1:
        tr = np.random.default_rng(seed).choice(tr, int(len(tr) * cfg['train_frac']), replace=False)
    train_cells = np.unique(pairs.cell_idx.values[tr])

    # per-layer matrices: optional XY removal, z-score using TRAIN cells only
    omics, dims = {}, {}
    for k in cfg['layers']:
        X = LAYER_ARR[k]
        if cfg['drop_xy'] and k in ('expr', 'cn'):
            X = X[:, [i for i, g in enumerate(LAYER_NAMES[k]) if g not in XY_GENES]]
        mu = np.nanmean(X[train_cells], 0); sd = np.nanstd(X[train_cells], 0) + 1e-6
        Xn = np.clip(np.nan_to_num((X - mu) / sd), -10, 10).astype('float32')
        omics[k] = torch.tensor(Xn, device=DEVICE); dims[k] = Xn.shape[1]

    sv = SEX_VEC.copy()
    if cfg['shuffle_sex']:                                            # break the sex signal, keep the marginals
        known = np.where(sv.sum(1) > 0)[0]
        sv[known] = sv[np.random.default_rng(seed + 1).permutation(known)]
    T_SEX = torch.tensor(sv, device=DEVICE)

    P_C = torch.tensor(pairs.cell_idx.values, device=DEVICE)
    P_D = torch.tensor(pairs.drug_idx.values, device=DEVICE)
    P_Y = torch.tensor(pairs.y.values, device=DEVICE)

    w = np.ones(len(pairs), 'float32')
    if cfg['reweight']:
        s = pairs.sex.values[tr]; cnt = {g: (s == g).sum() for g in ('Female', 'Male')}
        for g in ('Female', 'Male'):
            if cnt[g] > 0: w[pairs.sex.values == g] = len(tr) / (2 * cnt[g])
        w = w / w[tr].mean()
    P_W = torch.tensor(w, device=DEVICE)

    model = DRModel(dims, cfg['fusion'], cfg['sex_input']).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])

    def fwd(ix):
        c, d = P_C[ix], P_D[ix]
        return model({k: v[c] for k, v in omics.items()}, T_SEX[c], T_DFEAT[d], T_DADJ[d])

    def predict(ix):
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(ix), 2048):
                out.append(fwd(torch.as_tensor(ix[i:i + 2048], device=DEVICE)).cpu().numpy())
        return np.concatenate(out)

    best, best_state, bad = 1e9, None, 0
    for ep in range(cfg['epochs']):
        model.train(); perm = np.random.permutation(tr); tot = 0
        for i in range(0, len(perm), cfg['bs']):
            ix = torch.as_tensor(perm[i:i + cfg['bs']], device=DEVICE)
            loss = (P_W[ix] * (fwd(ix) - P_Y[ix]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)
        vr = float(np.sqrt(np.mean((predict(va) - pairs.y.values[va]) ** 2)))
        if verbose: print(f'  ep{ep:02d} train_mse={tot / len(tr):.4f} val_rmse={vr:.4f}')
        if vr < best - 1e-4: best, bad, best_state = vr, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg['patience']: break
    model.load_state_dict(best_state)

    df = pairs.iloc[te].copy(); df['pred'] = predict(te)
    rep = sex_report(df)
    row = dict(name=cfg['name'], fusion=cfg['fusion'], layers='+'.join(cfg['layers']), sex_input=cfg['sex_input'],
               reweight=cfg['reweight'], shuffle_sex=cfg['shuffle_sex'], drop_xy=cfg['drop_xy'],
               split=split_name, seed=seed, val_rmse=best,
               **{f'{g}_{m}': rep[g][m] for g in rep for m in ('n', 'rmse', 'pearson', 'spearman')})
    row['rmse_gap_F_minus_M'] = row['Female_rmse'] - row['Male_rmse']
    return row, df

RESULTS_CSV = OUT / 'results.csv'
def log_result(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=not RESULTS_CSV.exists(), index=False)

## 9. Smoke test (2 minutes) — run this first

In [ ]:
row, preds = run_experiment(dict(name='smoke', fusion='late', epochs=2, train_frac=0.1), 'cell_blind', seed=0)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in row.items()})

## 10. Baselines and fusion benchmark (cell-blind)
Each run is logged to `outputs/results.csv` on Drive. Use several seeds for real results (`SEEDS = [0,1,2,3,4]`).

In [ ]:
SPLIT = 'cell_blind'
SEEDS = [0]

BENCH = [
    dict(name='expr_only',    fusion='late',         layers=['expr']),               # baseline
    dict(name='deepcdr_like', fusion='deepcdr_like'),
    dict(name='early',        fusion='early'),
    dict(name='late',         fusion='late'),
    dict(name='attention',    fusion='attention'),
]
for cfg in BENCH:
    for sd in SEEDS:
        row, _ = run_experiment(cfg, SPLIT, seed=sd, verbose=False)
        log_result(row)
        print(f"{cfg['name']:14s} seed{sd}  all={row['all_rmse']:.4f}  F={row['Female_rmse']:.4f}  M={row['Male_rmse']:.4f}  gap={row['rmse_gap_F_minus_M']:+.4f}")

## 11. Sex-aware variants (all four components, per fusion) + ablations

In [ ]:
FUSIONS = ['early', 'late', 'attention', 'deepcdr_like']
AWARE = []
for f in FUSIONS:
    AWARE += [
        dict(name=f'{f}+sex_input',          fusion=f, sex_input=True),
        dict(name=f'{f}+reweight',           fusion=f, reweight=True),
        dict(name=f'{f}+sex_input+reweight', fusion=f, sex_input=True, reweight=True),
    ]
ABLATE = [   # on the late-fusion model, with sex input on
    dict(name='late+sex_input+SHUFFLED', fusion='late', sex_input=True, shuffle_sex=True),
    dict(name='late+drop_xy',            fusion='late', drop_xy=True),
    dict(name='late-no_expr',            fusion='late', layers=['mut', 'cn']),
    dict(name='late-no_cn',              fusion='late', layers=['expr', 'mut']),
    dict(name='late-no_mut',             fusion='late', layers=['expr', 'cn']),
]
for cfg in AWARE + ABLATE:
    for sd in SEEDS:
        row, _ = run_experiment(cfg, SPLIT, seed=sd, verbose=False)
        log_result(row)
        print(f"{cfg['name']:32s} seed{sd}  all={row['all_rmse']:.4f}  gap={row['rmse_gap_F_minus_M']:+.4f}")

## 12. Other split types (run the models you care about on each)

In [ ]:
for split in ['random', 'drug_blind'] + [f'tissue_blind_{k}' for k in range(5)]:
    for cfg in [dict(name='late', fusion='late'), dict(name='late+sex_input+reweight', fusion='late', sex_input=True, reweight=True)]:
        row, _ = run_experiment(cfg, split, seed=0, verbose=False); log_result(row)
        print(f"{split:18s} {cfg['name']:26s} all={row['all_rmse']:.4f} gap={row['rmse_gap_F_minus_M']:+.4f}")

## 13. Summaries, CIs, within-tissue comparison

In [ ]:
res = pd.read_csv(RESULTS_CSV)
res = res[res.name != 'smoke']
summ = (res.groupby(['split', 'name'])[['all_rmse', 'Female_rmse', 'Male_rmse', 'rmse_gap_F_minus_M',
                                        'all_pearson', 'Female_pearson', 'Male_pearson']]
          .agg(['mean', 'std']).round(4))
summ.to_csv(OUT / 'summary_by_model.csv')
summ

In [ ]:
# Bootstrap CI on the F-M RMSE gap for one model (resamples cell lines), plus a within-tissue comparison
row, preds = run_experiment(dict(name='late', fusion='late'), 'cell_blind', seed=0, verbose=False)
m, lo, hi = bootstrap_gap(preds, n_boot=200)
print(f'RMSE gap (F-M): {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]   (CI excluding 0 => detectable gap)')

MIN_PER_SEX = 5
rows = []
for t, d in preds.groupby('tissue'):
    f, mm = d[d.sex == 'Female'], d[d.sex == 'Male']
    if f.cell_idx.nunique() >= MIN_PER_SEX and mm.cell_idx.nunique() >= MIN_PER_SEX:
        rows.append(dict(tissue=t, n_F=f.cell_idx.nunique(), n_M=mm.cell_idx.nunique(),
                         rmse_F=metrics(f.y.values, f.pred.values)['rmse'], rmse_M=metrics(mm.y.values, mm.pred.values)['rmse']))
within = pd.DataFrame(rows).assign(gap=lambda d: d.rmse_F - d.rmse_M).sort_values('gap')
within.to_csv(OUT / 'within_tissue_gap.csv', index=False)
within

In [ ]:
# Plot gap by model
p = res[res.split == SPLIT].groupby('name')[['Female_rmse', 'Male_rmse']].mean()
p.plot.barh(figsize=(8, max(3, .35 * len(p))), color=['tab:red', 'tab:blue'])
plt.xlabel('test RMSE (lower is better)'); plt.title(f'{SPLIT}: female vs male lines'); plt.tight_layout()
plt.savefig(OUT / 'rmse_by_sex.png', dpi=150); plt.show()

## Notes / things to decide
- **"Remove X/Y genes" ablation:** `drop_xy` currently removes XIST + Y-linked genes only. To also remove X-linked genes you need a gene→chromosome table (e.g. from Ensembl BioMart or `mygene`); then extend `XY_GENES`.
- **X/Y gene module** from your proposal is not implemented yet. A natural version is a separate small encoder over the XY genes fed as an extra token/branch.
- **"Graph model" fusion** in the proposal is ambiguous (the drug side is already a graph in every model here). Decide what you want, e.g. a GNN over pathway/PPI-linked genes, before building it.
- **Leakage check:** feature selection is unsupervised and uses all cell lines; normalisation uses train cells only. Mention this in the methods.
- **Sex labels:** `sex` = metadata when present, else confident inference. `sex_mismatch` flags disagreements. Male lines with loss of Y can look "ambiguous" in inference, so interpret the mismatch rate carefully.
- Statistical power: use ≥5 seeds and the cell-line bootstrap before claiming any gap.
